In [87]:
# root directory setting

import os, sys

os.chdir("C:\\Code\\AI-Driven-Loan-Underwriting-Portfolio-Risk-Dashboard")    
sys.path.append(os.getcwd())              

In [88]:
# loading tables from databse to pandas dataframes

from db.database import Database
import os
from dotenv import load_dotenv

load_dotenv()

db = Database(
    server=os.getenv("DB_SERVER"),
    database=os.getenv("DB_DATABASE")
)

query_accepted = """SELECT TOP 300000 * FROM AcceptedCredit;"""
df_accepted = db.fetch(query_accepted)
print(df_accepted.shape)

query_rejected = """SELECT TOP 300000 * FROM RejectedCredit;"""
df_rejected = db.fetch(query_rejected)
print(df_rejected.shape)

C:\Code\AI-Driven-Loan-Underwriting-Portfolio-Risk-Dashboard\db\database.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn)


(20847, 151)
(200422, 9)


In [89]:
import pandas as pd
import numpy as np
import random

df_rejected["AmountRequested"] = (
    pd.to_numeric(df_rejected["AmountRequested"], errors="coerce")
    .astype(float)
)
df_rejected["ApplicationDate"] = pd.to_datetime(
    df_rejected["ApplicationDate"], errors="coerce"
)
df_rejected["LoanTitle"] = df_rejected["LoanTitle"].astype("string")
df_rejected["Risk_Score"] = pd.to_numeric(
    df_rejected["Risk_Score"], errors="coerce"
)
df_rejected["DebtToIncomeRatio"] = (
    df_rejected["DebtToIncomeRatio"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
    .str.replace(",", "", regex=False)
)
df_rejected["DebtToIncomeRatio"] = pd.to_numeric(
    df_rejected["DebtToIncomeRatio"], errors="coerce"
).astype(float)
df_rejected["ZipCode"] = (
    df_rejected["ZipCode"]
    .astype(str)
    .str.extract(r"(\d{3})")[0]
)
df_rejected["ZipCode"] = pd.to_numeric(
    df_rejected["ZipCode"], errors="coerce"
).astype("Int64")
df_rejected["ZipCode"] = df_rejected["ZipCode"].astype("category")
df_rejected["State"] = df_rejected["State"].astype("category")
df_rejected["PolicyCode"] = df_rejected["PolicyCode"].astype("category")

In [90]:
# correcting data types in df_accepted

import pandas as pd

numeric_cols = [
    'loan_amnt','funded_amnt','funded_amnt_inv','int_rate','installment',
    'annual_inc','dti','delinq_2yrs','fico_range_low','fico_range_high',
    'inq_last_6mths','mths_since_last_delinq','open_acc','pub_rec',
    'revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
    'total_pymnt','total_pymnt_inv','total_rec_prncp','total_rec_int',
    'total_rec_late_fee','recoveries','collection_recovery_fee',
    'last_pymnt_amnt','last_fico_range_high','last_fico_range_low',
    'collections_12_mths_ex_med','acc_now_delinq','tot_coll_amt',
    'tot_cur_bal','open_acc_6m','open_act_il','open_il_12m','open_il_24m',
    'total_bal_il','il_util','open_rv_12m','open_rv_24m','max_bal_bc',
    'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m',
    'acc_open_past_24mths','avg_cur_bal','bc_open_to_buy','bc_util',
    'chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl',
    'mort_acc','mths_since_recent_bc','num_accts_ever_120_pd',
    'num_actv_bc_tl','num_actv_rev_tl','num_bc_sats','num_bc_tl',
    'num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0',
    'num_sats','pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim',
    'total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit'
]
df_accepted[numeric_cols] = df_accepted[numeric_cols].apply(pd.to_numeric, errors='coerce')


date_cols = [
    'issue_d', 'earliest_cr_line', 'last_pymnt_d',
    'next_pymnt_d', 'last_credit_pull_d', 'sec_app_earliest_cr_line'
]
for col in date_cols:
    df_accepted[col] = pd.to_datetime(df_accepted[col], format='%b-%Y', errors='coerce')


category_cols = [
    'term','grade','sub_grade','emp_length','home_ownership','verification_status',
    'loan_status','pymnt_plan','purpose','zip_code','addr_state',
    'application_type','initial_list_status','hardship_flag','debt_settlement_flag'
]
for col in category_cols:
    df_accepted[col] = df_accepted[col].astype('category')


text_cols = ['desc','title','url','emp_title','hardship_type','hardship_reason','hardship_status']
for col in text_cols:
    df_accepted[col] = df_accepted[col].astype('string')


In [91]:
import pandas as pd

# Show full DataFrame output (no cropping)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# --- Missing Values Report ---
def null_report(df):
    return (
        pd.DataFrame({
            'null_count': df.isna().sum(),
            'null_percent': (df.isna().mean() * 100).round(2)
        })
        .sort_values('null_percent', ascending=False)
    )

# --- Unique Values Report ---
def unique_report(df):
    return (
        pd.DataFrame({
            'unique_count': df.nunique(),
        })
        .sort_values('unique_count', ascending=False)
    )

# --- Combined Full Report ---
def full_report(df):
    nulls = df.isna().sum()
    null_percent = (df.isna().mean() * 100).round(2)
    uniques = df.nunique()

    report = pd.DataFrame({
        'null_count': nulls,
        'null_percent': null_percent,
        'unique_count': uniques
    }).sort_values('null_percent', ascending=False)

    return report


# ---- PRINT ALL REPORTS -----

print("\n===== ACCEPTED: FULL REPORT =====")
print(full_report(df_accepted))

print("\n===== REJECTED: FULL REPORT =====")
print(full_report(df_rejected))


===== ACCEPTED: FULL REPORT =====
                                            null_count  null_percent  unique_count
member_id                                        20847        100.00             0
il_util                                          20847        100.00             0
verification_status_joint                        20847        100.00             0
tot_cur_bal                                      20847        100.00             0
open_il_12m                                      20847        100.00             0
open_act_il                                      20847        100.00             0
open_rv_24m                                      20847        100.00             0
avg_cur_bal                                      20847        100.00             0
bc_open_to_buy                                   20847        100.00             0
bc_util                                          20847        100.00             0
total_rev_hi_lim                                 208

In [92]:
# Dropping columns with 100% nulls and constant columns

cols_100_null = full_report(df_accepted).query("null_percent > 70").index.tolist()
df_accepted = df_accepted.drop(columns=cols_100_null)

cols_100_null = full_report(df_rejected).query("null_percent > 70").index.tolist()
df_rejected = df_rejected.drop(columns=cols_100_null)

const_cols = full_report(df_accepted).query("unique_count <= 1").index.tolist()
df_accepted = df_accepted.drop(columns=const_cols)

const_cols = full_report(df_rejected).query("unique_count <= 1").index.tolist()
df_rejected = df_rejected.drop(columns=const_cols)

In [93]:
df_accepted['emp_length']

0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
5              NaN
6              NaN
7              NaN
8              NaN
9              NaN
10             NaN
11             NaN
12             NaN
13             NaN
14             NaN
15             NaN
16             NaN
17             NaN
18             NaN
19             NaN
20         7 years
21        < 1 year
22        < 1 year
23        < 1 year
24        < 1 year
25        < 1 year
26        < 1 year
27        < 1 year
28        < 1 year
29        < 1 year
30        < 1 year
31        < 1 year
32        < 1 year
33        < 1 year
34        < 1 year
35        < 1 year
36        < 1 year
37       10+ years
38        < 1 year
39         5 years
40         2 years
41       10+ years
42         7 years
43         2 years
44         8 years
45         3 years
46             NaN
47          1 year
48         4 years
49         3 years
50         4 years
51         3 years
52        < 

In [94]:
import pandas as pd

# Show full DataFrame output (no cropping)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# --- Missing Values Report ---
def null_report(df):
    return (
        pd.DataFrame({
            'null_count': df.isna().sum(),
            'null_percent': (df.isna().mean() * 100).round(2)
        })
        .sort_values('null_percent', ascending=False)
    )

# --- Unique Values Report ---
def unique_report(df):
    return (
        pd.DataFrame({
            'unique_count': df.nunique(),
        })
        .sort_values('unique_count', ascending=False)
    )

# --- Combined Full Report ---
def full_report(df):
    nulls = df.isna().sum()
    null_percent = (df.isna().mean() * 100).round(2)
    uniques = df.nunique()

    report = pd.DataFrame({
        'null_count': nulls,
        'null_percent': null_percent,
        'unique_count': uniques
    }).sort_values('null_percent', ascending=False)

    return report


# ---- PRINT ALL REPORTS -----

print("\n===== ACCEPTED: FULL REPORT =====")
print(full_report(df_accepted))

print("\n===== REJECTED: FULL REPORT =====")
print(full_report(df_rejected))


===== ACCEPTED: FULL REPORT =====
                         null_count  null_percent  unique_count
mths_since_last_delinq        12578         60.33            89
desc                           4331         20.78         16247
pub_rec_bankruptcies           1398          6.71             3
emp_title                      1223          5.87         16037
emp_length                      386          1.85            11
tax_liens                       138          0.66             2
revol_util                      114          0.55          1064
last_pymnt_d                     78          0.37           101
delinq_2yrs                      62          0.30            11
inq_last_6mths                   62          0.30            28
earliest_cr_line                 62          0.30           488
delinq_amnt                      62          0.30             3
total_acc                        62          0.30            78
open_acc                         62          0.30            43
pub_r

In [95]:
# ---------- Handle mths_since_last_delinq (60% missing) ----------
df_accepted["has_delinq"] = df_accepted["mths_since_last_delinq"].notna().astype(int)
df_accepted["mths_since_last_delinq"] = df_accepted["mths_since_last_delinq"].fillna(999)

# ---------- desc (20.78% missing → fill text) ----------
df_accepted["desc"] = df_accepted["desc"].fillna("No Description")

# ---------- emp_title (5.87% missing) ----------
df_accepted["emp_title"] = df_accepted["emp_title"].fillna("Unknown")

# ---------- emp_length (1.85% missing) ----------
df_accepted["emp_length"] = df_accepted["emp_length"].fillna("< 1 year")

# ---------- tax_liens (0.66% missing, binary?) ----------
df_accepted["tax_liens"] = df_accepted["tax_liens"].fillna(0)

# ---------- revol_util (0.55% missing) ----------
df_accepted["revol_util"] = df_accepted["revol_util"].fillna(df_accepted["revol_util"].median())

# ---------- last_pymnt_d (0.37% missing → date) ----------
df_accepted["last_pymnt_d"] = df_accepted["last_pymnt_d"].fillna(df_accepted["last_pymnt_d"].max())

# ---------- Small-missing numeric columns (~0.16–0.3%) ----------
numeric_small_missing = [
    "delinq_2yrs", "inq_last_6mths", "earliest_cr_line",
    "delinq_amnt", "total_acc", "open_acc", "pub_rec", "acc_now_delinq",
    "annual_inc", "total_pymnt", "funded_amnt", "funded_amnt_inv",
    "loan_amnt", "installment", "int_rate", "revol_bal",
    "last_pymnt_amnt", "fico_range_low", "fico_range_high",
    "last_fico_range_low", "last_fico_range_high",
    "total_rec_int", "total_rec_late_fee", "total_rec_prncp",
    "recoveries", "total_pymnt_inv", "dti"
]

for col in numeric_small_missing:
    if col in df_accepted.columns:
        df_accepted[col] = df_accepted[col].fillna(df_accepted[col].median())

# ---------- Categorical small-missing ----------
cat_small_missing = [
    "purpose", "loan_status", "home_ownership", "verification_status",
    "sub_grade", "grade", "addr_state", "term", "debt_settlement_flag"
]

for col in cat_small_missing:
    if col in df_accepted.columns:
        df_accepted[col] = df_accepted[col].dropna()

# ---------- Date fields ----------
date_small_missing = ["issue_d", "last_credit_pull_d"]
for col in date_small_missing:
    if col in df_accepted.columns:
        df_accepted[col] = df_accepted[col].fillna(df_accepted[col].max())

# ---------- Text fields ----------
df_accepted["title"] = df_accepted["title"].fillna("Unknown Title")
df_accepted["url"] = df_accepted["url"].fillna("Unknown URL")

# ---------- Zip code ----------
df_accepted["zip_code"] = df_accepted["zip_code"].cat.add_categories(["000xx"])
df_accepted["zip_code"] = df_accepted["zip_code"].fillna("000xx")


In [ ]:
# ---------- ZipCode (9.36% missing) ----------
df_rejected["ZipCode"] = df_rejected["ZipCode"].cat.add_categories(["000xx"])
df_rejected["ZipCode"] = df_rejected["ZipCode"].fillna("000xx")

# ---------- Risk_Score (8.86% missing, numeric) ----------
df_rejected["Risk_Score"] = df_rejected["Risk_Score"].fillna(df_rejected["Risk_Score"].median())

# ---------- DebtToIncomeRatio (0.15%) ----------
df_rejected["DebtToIncomeRatio"] = df_rejected["DebtToIncomeRatio"].fillna(df_rejected["DebtToIncomeRatio"].median())

# ---------- LoanTitle (0.01% missing) ----------
df_rejected["LoanTitle"] = df_rejected["LoanTitle"].fillna("Unknown")

# ---------- State (0.01% missing) ----------
df_rejected["State"] = df_rejected["State"].cat.add_categories("Unknown")
df_rejected["State"] = df_rejected["State"].fillna("Unknown")